# 2. Data curation: activation-loop filter benchmark <a id="2"></a>

Compare three ways to identify DFG–APE activation loops on the same starting set
`Results/InterPro_KLIFS_protein_chains/`:

1. **Sequence motif** (basis) — SEQRES/ATOM string contains `DFG` and `APE` (`copy_filtered_pdbs`)
2. **MUSCLE MSA** — whole-chain amino-acid MSA; non-gapped DFG/APE columns vs `6UAN_chainD`
3. **KLIFS + HMMER** — KLIFS DFG/APE for curated chains; Pfam/HMMER for the remainder (Workflow 1 §2.1)

Outputs (pass sets, loop TSVs, plots, HTML MSAs) live under
`Results/Experiments/loop_filter_benchmark/` and do **not** overwrite production
`Results/motif_filtered_chains/`.


## Table of contents

- [2.0 Setup](#setup)
  - [Pseudokinase exclusion list](#pseudokinase-exclusion-list)
- [2.1 Method A — sequence motif (basis)](#motif)
- [2.2 Method B — MUSCLE MSA](#muscle)
- [2.3 Method C — KLIFS + HMMER](#klifshmmer)
- [2.4 Loop-length histograms](#hist)
- [2.5 DFG–APE MSA HTML (per method)](#msa)


## Backend map

How this notebook connects to `workflow/` modules:

![Backend map](images/backend_maps/03c-LoopFilterBenchmark.svg)

<!-- mermaid source (GitHub does not render mermaid in .ipynb; SVG above is for GitHub):
```mermaid
flowchart LR
  nb["03c-LoopFilterBenchmark"]
  m0["workflow.align_MSA"]
  nb --> m0
  m1["workflow.hmmer_diagnostics"]
  nb --> m1
  m2["workflow.kinaseGroupLabelling"]
  nb --> m2
  m3["workflow.klifs_filter"]
  nb --> m3
  m4["workflow.loop_filter_benchmark"]
  nb --> m4
  m5["workflow.msa_visualiser"]
  nb --> m5
  m6["workflow.utilities"]
  nb --> m6
```
-->


## 2.0 Setup  <a id="setup"></a>


In [ ]:
import warnings
warnings.filterwarnings("ignore")
import os
import shutil
import pandas as pd
from IPython.display import display, HTML, FileLink

from workflow.align_MSA import AlignmentMUSCLE
from workflow.hmmer_diagnostics import HMMERDiagnostics
from workflow.kinaseGroupLabelling import KinaseGroupLabeller
from workflow.klifs_filter import KLIFSOverlap
from workflow.loop_filter_benchmark import (
    extract_dfg_ape_loops_from_pdbs,
    plot_loop_length_histograms,
    write_dfg_ape_msa_html,
    muscle_column_aligned_loops,
    summarize_method_counts,
    copy_basenames_to_dir,
)
from workflow.msa_visualiser import MSAVisualiser
from workflow.utilities import count_pdb_files, copy_filtered_pdbs, clear_and_make


### Pseudokinase exclusion list <a id="pseudokinase-exclusion-list"></a>

Annotate the KLIFS-merged chain set once so Methods A and C share the same
`excluded_pseudokinase_basenames.txt`.


In [ ]:
lab = KinaseGroupLabeller()
annot_all = lab.annotate_dataset_chains_with_kinome(
    INPUT_DIR,
    output_csv=ANNOT_CSV,
    exclude_pseudokinase=True,
)
# excluded_pseudokinase_basenames.txt is written beside ANNOT_CSV (= PSEUDO_FILE)
assert os.path.isfile(PSEUDO_FILE) or os.path.isfile(
    os.path.join(EXPERIMENT_DIR, "excluded_pseudokinase_basenames.txt")
)
if not os.path.isfile(PSEUDO_FILE):
    import shutil
    shutil.copy2(
        os.path.join(EXPERIMENT_DIR, "excluded_pseudokinase_basenames.txt"),
        PSEUDO_FILE,
    )

n_pseudo = sum(1 for line in open(PSEUDO_FILE) if line.strip()) if os.path.isfile(PSEUDO_FILE) else 0
print(f"Annotation → {ANNOT_CSV} ({len(annot_all)} rows)")
print(f"Pseudokinase exclusion → {PSEUDO_FILE} ({n_pseudo} basenames)")
display(annot_all.head())


## 2.1 Method A — sequence motif (basis)  <a id="motif"></a>

Keep chains whose extracted sequence contains both `DFG` and `APE` (pseudokinases excluded).
Then extract DFG→APE loop sequences (residues between the motifs).


In [ ]:
clear_and_make(MOTIF_PASS)
valid_pdbs, invalid_pdbs = copy_filtered_pdbs(
    source_dir=INPUT_DIR,
    target_dir=MOTIF_PASS,
    motifs=["DFG", "APE"],
    excluded_basenames_file=PSEUDO_FILE,
)
print(f"Motif pass: {len(valid_pdbs)}  fail: {len(invalid_pdbs)}")
print(f"Copied to {MOTIF_PASS}: {count_pdb_files(MOTIF_PASS)} PDBs")

motif_loop_df = extract_dfg_ape_loops_from_pdbs(
    MOTIF_PASS,
    loop_tsv=MOTIF_LOOP_TSV,
)
display(motif_loop_df.head())


## 2.2 Method B — MUSCLE MSA  <a id="muscle"></a>

Whole-dataset MUSCLE v5 MSA; classify chains by non-gapped DFG/APE columns in the
`6UAN_chainD` reference. Loop TSV and passing basenames are written under `muscle/`.


In [ ]:
msa = AlignmentMUSCLE(
    muscle_bin=MUSCLE_BIN,
    results_dir=MUSCLE_DIR,
)

muscle_results = msa.run(
    input_dir=INPUT_DIR,
    reference_pdb=REFERENCE_PDB,
    reference_name=REFERENCE_NAME,
    generate_html=False,  # benchmark HTML written in §5
)

n_copied = copy_basenames_to_dir(
    INPUT_DIR,
    muscle_results["msa_passing"],
    MUSCLE_PASS,
)
# Align loop TSV path used by helpers (AlignmentMUSCLE already wrote it)
assert os.path.isfile(msa.loop_tsv_path), msa.loop_tsv_path
if os.path.abspath(msa.loop_tsv_path) != os.path.abspath(MUSCLE_LOOP_TSV):
    import shutil
    shutil.copy2(msa.loop_tsv_path, MUSCLE_LOOP_TSV)

print(f"MUSCLE passing: {len(muscle_results['msa_passing'])}")
print(f"Copied to {MUSCLE_PASS}: {n_copied} PDBs")
print(f"Loop TSV → {MUSCLE_LOOP_TSV}")


## 2.3 Method C — KLIFS + HMMER  <a id="klifshmmer"></a>

KLIFS DFG/APE for chains in KLIFS; HMMER (`PF00069`) for the remainder. All filter
outputs are scoped to `klifs_hmmer/` so production motif directories are untouched.

Reuse existing `Results/KLIFS/` inventory/catalog caches when present by pointing
`klifs_dir` at a benchmark subdirectory while allowing inventory rebuild from `INPUT_DIR`.


In [ ]:
KH_KLIFS_DIR = os.path.join(KH_DIR, "KLIFS")
KH_HMMER_DIR = os.path.join(KH_DIR, "HMMER")
os.makedirs(KH_KLIFS_DIR, exist_ok=True)
os.makedirs(KH_HMMER_DIR, exist_ok=True)

# Reuse heavy caches from the main August Results/KLIFS if available
import shutil
for fname in (
    "klifs_full_catalog_pdb_chain.tsv",
    "klifs_full_catalog_summary.json",
    "klifs_structure_inventory.tsv",
    "match_residues_cache.tsv",
):
    src = os.path.join("Results/KLIFS", fname)
    dst = os.path.join(KH_KLIFS_DIR, fname)
    if os.path.isfile(src) and not os.path.isfile(dst):
        shutil.copy2(src, dst)
        print(f"Reused cache {fname}")

# Empty SM dir is fine — ligand copy is best-effort
KH_SM_DIR = os.path.join(KH_DIR, "small_molecules_unused")
os.makedirs(KH_SM_DIR, exist_ok=True)

# Fresh combined loop TSV for this experiment
if os.path.isfile(KH_LOOP_TSV):
    os.remove(KH_LOOP_TSV)

klifs = KLIFSOverlap(
    input_dir=INPUT_DIR,
    small_molecule_dir=KH_SM_DIR,
    klifs_dir=KH_KLIFS_DIR,
    target_protein_dir=KH_PASS,
    target_ligand_dir=os.path.join(KH_DIR, "passing_sm"),
    loop_tsv=KH_LOOP_TSV,
    pseudo_basenames_path=PSEUDO_FILE,
)

print("Building KLIFS inventory …")
klifs.build_inventory(force=False)
print("Running KLIFS DFG/APE filter …")
klifs_results = klifs.run_filter()
print("KLIFS filter done.")


In [ ]:
hmmer = HMMERDiagnostics(
    chain_dir=INPUT_DIR,
    non_klifs_list=klifs.hmmer_input_basenames_txt,
    hmmer_dir=KH_HMMER_DIR,
    hmm_path=os.path.join(KH_HMMER_DIR, "Pkinase.hmm"),
    interpro_spans_tsv=os.path.join(KH_HMMER_DIR, "interpro_domain_spans.tsv"),
    pseudo_basenames_path=PSEUDO_FILE,
    apply_pseudokinase_exclusion=True,
    target_chains_dir=KH_PASS,
    loop_tsv=KH_LOOP_TSV,
)

# Reuse HMM if already downloaded under Results/HMMER
_main_hmm = "Results/HMMER/Pkinase.hmm"
if os.path.isfile(_main_hmm) and not os.path.isfile(hmmer.hmm_path):
    shutil.copy2(_main_hmm, hmmer.hmm_path)
    print(f"Reused HMM → {hmmer.hmm_path}")

print("Running HMMER diagnostics …")
hmmer_diagnostics = hmmer.run(copy_passing=True)
display(hmmer_diagnostics.head())

print(f"KLIFS+HMMER combined pass dir: {KH_PASS} ({count_pdb_files(KH_PASS)} PDBs)")
print(f"Combined loop TSV → {KH_LOOP_TSV}")
if os.path.isfile(KH_LOOP_TSV):
    print(pd.read_csv(KH_LOOP_TSV, sep="\t").shape)


## 2.4 Loop-length histograms  <a id="hist"></a>

Distribution of DFG–APE loop lengths for each method’s pass set.


In [ ]:
method_tsvs = {
    "Motif (DFG+APE)": MOTIF_LOOP_TSV,
    "MUSCLE MSA": MUSCLE_LOOP_TSV,
    "KLIFS+HMMER": KH_LOOP_TSV,
}
for label, path in method_tsvs.items():
    if not os.path.isfile(path):
        raise FileNotFoundError(f"Missing loop TSV for {label}: {path}")

summary_df = summarize_method_counts(method_tsvs)
display(summary_df)
summary_df.to_csv(os.path.join(EXPERIMENT_DIR, "method_summary.csv"), index=False)

plot_loop_length_histograms(method_tsvs, HIST_PNG, show=True)


## 2.5 DFG–APE MSA HTML (per method)  <a id="msa"></a>

One Workflow1-style HTML file **per method** (passers only). Each file has Full and
DFG–APE tabs (Clustal amino-acid colours). MUSCLE uses column-aligned gapped loops
from the MSA when available; motif and KLIFS+HMMER use left-padded loop sequences
from their TSVs.

Additionally, Method C writes the full Workflow1 KLIFS/HMMER MSA suite under
`klifs_hmmer/msa/`.


In [ ]:
# Motif
write_dfg_ape_msa_html(
    MOTIF_LOOP_TSV,
    MSA_HTML["motif"],
    title="Method A — sequence motif DFG–APE",
    category="motif_passed",
)

# MUSCLE — column-aligned loops when msa_dict is in memory
aligned = None
full_msa = None
try:
    aligned = muscle_column_aligned_loops(
        muscle_results["msa_dict"],
        muscle_results["msa_passing"],
        muscle_results["dfg_cols"],
        muscle_results["ape_cols"],
    )
    full_msa = {
        n: muscle_results["msa_dict"][n]
        for n in muscle_results["msa_passing"]
        if n in muscle_results["msa_dict"]
    }
except NameError:
    print("muscle_results not in memory — writing MUSCLE HTML from loop TSV only")

write_dfg_ape_msa_html(
    MUSCLE_LOOP_TSV,
    MSA_HTML["muscle"],
    title="Method B — MUSCLE MSA DFG–APE",
    category="muscle_passed",
    aligned_seqs=aligned,
    full_seqs=full_msa,
)

# KLIFS+HMMER combined passers (loop TSV)
write_dfg_ape_msa_html(
    KH_LOOP_TSV,
    MSA_HTML["klifs_hmmer"],
    title="Method C — KLIFS+HMMER DFG–APE",
    category="klifs_hmmer_passed",
)

print("Benchmark HTML files:")
for k, p in MSA_HTML.items():
    print(f"  {k:12s} → {p}  ({'ok' if os.path.isfile(p) else 'MISSING'})")
    if os.path.isfile(p):
        display(FileLink(p))


In [ ]:
# Full Workflow1-style MSA suite for Method C (optional but matches §2.1.5)
KH_MSA_OUT = os.path.join(KH_DIR, "msa")
os.makedirs(KH_MSA_OUT, exist_ok=True)

# match_residues cache required for KLIFS full panel
try:
    klifs.build_match_residues_cache()
except Exception as exc:
    print(f"match_residues cache note: {exc}")

vis = MSAVisualiser(
    klifs_dir=KH_KLIFS_DIR,
    hmmer_dir=KH_HMMER_DIR,
    chain_dir=INPUT_DIR,
    motif_dir=KH_PASS,
    output_dir=KH_MSA_OUT,
    annot_csv=ANNOT_CSV,
    loop_tsv=KH_LOOP_TSV,
    hmm_path=os.path.join(KH_HMMER_DIR, "Pkinase.hmm"),
    hmmer_fasta=os.path.join(KH_HMMER_DIR, "non_klifs_sequences.fasta"),
    diagnostics_tsv=os.path.join(KH_HMMER_DIR, "hmmer_diagnostics.tsv"),
    failures_txt=os.path.join(KH_HMMER_DIR, "hmmer_failures.txt"),
    klifs_failures_txt=os.path.join(KH_KLIFS_DIR, "klifs_filter_failures.txt"),
)
try:
    msa_paths = vis.run()
    print("\nMSAVisualiser outputs:")
    for label, path in msa_paths.items():
        print(f"  {label:30s} → {path}")
except Exception as exc:
    print(f"MSAVisualiser.run() skipped/failed: {exc}")
    print("Benchmark per-method DFG–APE HTML files in §5 above are still available.")
